In [ ]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.lstm_dataset import LSTMDataset
from src.lstm_model import DrowsinessLSTM

In [ ]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
windows_df = pd.read_csv(
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "windows.csv"
)

with open(
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "folds.json"
) as f:
    folds = json.load(f)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

In [ ]:
fold = folds[0]

In [ ]:
windows_df["subject"] = (
    windows_df["subject"]
    .astype(str)
    .str.zfill(2)
)

In [ ]:
train_df = windows_df[
    windows_df["subject"].isin(
        fold["train_subjects"]
    )
]

val_df = windows_df[
    windows_df["subject"].isin(
        fold["val_subjects"]
    )
]

print(len(train_df))
print(len(val_df))

In [ ]:
print(windows_df["subject"].unique()[:10])

In [ ]:
train_dataset = LSTMDataset(train_df)
val_dataset = LSTMDataset(val_df)

print(len(train_dataset))
print(len(val_dataset))

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

In [ ]:
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)

In [ ]:
model = DrowsinessLSTM().to(device)

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

In [ ]:
with torch.no_grad():
    out = model(x.to(device))

print(out.shape)

In [ ]:
best_f1 = 0
patience = 10
counter = 0

for epoch in range(50):

    print(f"\nEpoch {epoch+1}/50")

    ################
    # TRAIN
    ################

    model.train()

    train_preds = []
    train_labels = []

    for x, y in tqdm(train_loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(
            logits,
            y
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        preds = logits.argmax(1)

        train_preds.extend(
            preds.cpu().numpy()
        )

        train_labels.extend(
            y.cpu().numpy()
        )

    train_acc = accuracy_score(
        train_labels,
        train_preds
    )

    train_f1 = f1_score(
        train_labels,
        train_preds,
        average="macro"
    )

    ################
    # VALIDATION
    ################

    model.eval()

    val_preds = []
    val_labels = []

    with torch.no_grad():

        for x, y in val_loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            preds = logits.argmax(1)

            val_preds.extend(
                preds.cpu().numpy()
            )

            val_labels.extend(
                y.cpu().numpy()
            )

    val_acc = accuracy_score(
        val_labels,
        val_preds
    )

    val_f1 = f1_score(
        val_labels,
        val_preds,
        average="macro"
    )

    print(
        f"Train Acc: {train_acc:.4f}"
    )
    print(
        f"Train F1: {train_f1:.4f}"
    )

    print(
        f"Val Acc: {val_acc:.4f}"
    )
    print(
        f"Val F1: {val_f1:.4f}"
    )

    print(
        classification_report(
            val_labels,
            val_preds,
            target_names=[
                "Alert",
                "Low Vigilant",
                "Drowsy"
            ],
            digits=4
        )
    )

    if val_f1 > best_f1:

        best_f1 = val_f1
        counter = 0

        torch.save(
            model.state_dict(),
            PROJECT_ROOT
            / "models"
            / "lstm_fold1.pth"
        )

        print("✅ Saved")

    else:
        counter += 1
        print(
            f"No improvement ({counter}/10)"
        )

        if counter >= patience:
            print("Early stopping")
            break

In [ ]:
print(train_df.shape)
print(val_df.shape)

print(train_loader.batch_size)
print(val_loader.batch_size)

print(criterion)
print(optimizer)

print(model)

print(fold["val_subjects"])